# E3SM S2S Multi-Experiment Skill & RMSE Comparison (Weeks 1 to 8)

This notebook compares **subseasonal prediction performance across initialization experiments** across **Weeks 1 to 8** (Days 1–56).

### Comparative Questions
1. **Ocean Initialization Impact**: Does 4DEnVar ocean initialization reduce atmospheric and SST errors faster than JRA55-forced FOSIRL during Weeks 1–4?
2. **Relative Skill Gain**: Where globally does the greatest error reduction occur across subseasonal weekly leads ($L=1..8$)?
3. **Skill Difference**: $\Delta\text{RMSE} = \text{RMSE}_{\text{4DEnVar}} - \text{RMSE}_{\text{JRA55}}$ and $\Delta\text{ACC} = \text{ACC}_{\text{4DEnVar}} - \text{ACC}_{\text{JRA55}}$.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

repo_root = Path.cwd()
while repo_root.parent != repo_root and not (repo_root / "esp_lab").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from esp_lab.paths import figure_output_dir
from esp_lab.diagnostics.s2s_core import (
    S2S_WEEKLY_WINDOWS,
    WEEK_NAMES,
    WEEK_LABELS,
    get_weekly_window,
    compute_weekly_anomalies,
    compute_weekly_rmse,
    compute_weekly_acc,
    paired_acc_difference,
)
from esp_lab.diagnostics.s2s_io import (
    DEFAULT_DATA_DIR,
    load_s2s_campaign_weekly,
)

print("S2S Cross-Experiment Comparative Module Loaded.")

## Configuration and Control Panel

In [ ]:
# =============================================================================
# USER CONTROL PANEL — S2S COMPARATIVE SKILL (WEEKS 1 TO 8)
# =============================================================================

FIELD = "TREFHT"
COMPONENT = "atm"
GRID = "180x360_aave"

LEAD_WEEKS = list(range(1, 9))
INIT_YEARS = list(range(1980, 1987))
INIT_MONTHS = [5, 11]
MEMBERS = [f"EN{i:02d}" for i in range(10)]

# Comparative Experiment Pair
TEST_CASE = "E3SM-4DEnVarOcn"
REF_CASE = "E3SM-JRA55_FOSIRL"

E3SM_CASES = {
    "E3SM-4DEnVarOcn": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
        "label": "4DEnVar Ocean Init",
    },
    "E3SM-JRA55_FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "label": "JRA55-FOSIRL Ocean Init",
    },
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "label": "Reanalysis (BruteForce)",
    },
}

FIGURE_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR = figure_output_dir("s2s_skill", COMPONENT, "weekly_compare", root=FIGURE_ROOT)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"Comparing: {TEST_CASE} vs {REF_CASE}")
print(f"Weekly Leads: {LEAD_WEEKS}")
print(f"Figure Output Directory: {FIGURE_OUTDIR}")

## Step 1 — Load Hindcasts and Compute Comparative Metrics

In [ ]:
%%time
model_weekly = {}

for case_key in [TEST_CASE, REF_CASE, "E3SM-Reanalysis"]:
    model_weekly[case_key] = {}
    prefix = E3SM_CASES[case_key]["case_prefix"]
    for m in INIT_MONTHS:
        try:
            da = load_s2s_campaign_weekly(
                data_root=DEFAULT_DATA_DIR,
                case_prefix=prefix,
                years=INIT_YEARS,
                init_month=m,
                members=MEMBERS,
                field=FIELD,
                component=COMPONENT,
                grid=GRID,
                weeks=LEAD_WEEKS,
                verbose=False,
            )
            model_weekly[case_key][m] = da
        except Exception:
            pass

# Compute RMSE relative to Reanalysis
rmse_diff_by_month = {}
benchmark = "E3SM-Reanalysis"

for m in INIT_MONTHS:
    if m in model_weekly[TEST_CASE] and m in model_weekly[REF_CASE] and m in model_weekly[benchmark]:
        obs = model_weekly[benchmark][m].mean("M", skipna=True)
        test_em = model_weekly[TEST_CASE][m].mean("M", skipna=True)
        ref_em = model_weekly[REF_CASE][m].mean("M", skipna=True)
        rmse_test = np.sqrt(((test_em - obs) ** 2).mean("Y", skipna=True))
        rmse_ref = np.sqrt(((ref_em - obs) ** 2).mean("Y", skipna=True))
        # Delta RMSE = RMSE(Test) - RMSE(Ref); negative means Test has lower error (better)
        diff = rmse_test - rmse_ref
        diff.name = "delta_rmse"
        rmse_diff_by_month[m] = diff
        print(f"Delta RMSE computed for Month {m:02d}")

print("Comparative calculations complete.")

## Step 2 — Spatial Difference Maps: ΔRMSE Across Weeks 1 to 8

Blue shading indicates $\text{RMSE}_{\text{4DEnVar}} < \text{RMSE}_{\text{JRA55}}$ (skill improvement by 4DEnVar ocean initialization).

In [ ]:
%%time
def plot_delta_rmse_maps(diff_da, test_label, ref_label, init_month, field_name):
    fig, axes = plt.subplots(
        nrows=2, ncols=4, figsize=(20, 9),
        subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)}
    )
    axes = axes.flatten()
    max_amp = float(np.abs(diff_da).quantile(0.98))
    levels = np.linspace(-max_amp, max_amp, 21)
    month_name = {5: "May", 11: "November"}.get(init_month, f"Month {init_month}")

    for idx, w in enumerate(range(1, 9)):
        ax = axes[idx]
        ax.coastlines(linewidth=0.8, color="0.2")
        ax.set_global()
        if w in diff_da.L.values:
            dw = diff_da.sel(L=w)
            cf = ax.contourf(
                dw.lon, dw.lat, dw,
                levels=levels, cmap="coolwarm", extend="both",
                transform=ccrs.PlateCarree()
            )
        w_def = get_weekly_window(w)
        ax.set_title(w_def.label, fontsize=12, fontweight="bold")

    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
    cbar = fig.colorbar(cf, cax=cbar_ax, orientation="horizontal")
    cbar.set_label(f"\u0394RMSE: {test_label} minus {ref_label}", fontsize=12)

    fig.suptitle(
        f"{field_name} Subseasonal \u0394RMSE (Weeks 1–8): {test_label} vs {ref_label}\nInitialized {month_name}",
        fontsize=16, fontweight="bold", y=0.98
    )
    plt.subplots_adjust(bottom=0.12, top=0.92, hspace=0.15, wspace=0.08)

    out_path = FIGURE_OUTDIR / f"delta_rmse_{test_label}_vs_{ref_label}_{field_name}_init{init_month:02d}_w1_w8.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Figure saved: {out_path}")
    plt.show()

for m in rmse_diff_by_month:
    plot_delta_rmse_maps(
        rmse_diff_by_month[m],
        test_label=TEST_CASE,
        ref_label=REF_CASE,
        init_month=m,
        field_name=FIELD,
    )

## Validation & Integrity Check

In [ ]:
for m in rmse_diff_by_month:
    diff_da = rmse_diff_by_month[m]
    assert "L" in diff_da.dims, "L dimension missing in diff_da"
    assert len(diff_da.L) == 8, f"Expected 8 weekly leads, found {len(diff_da.L)}"

print("Validation SUCCESS: All 8 weekly comparative leads verified.")